# Notebook 03: Researcher Extraction

**Input:** `data/papers.csv`  
**Output:** `data/researchers.csv`

This notebook processes each paper from the ArXiv search results and uses LLM to extract researcher information including names, affiliations, research focus, and seniority levels.

## Setup and Imports

In [ ]:
import sys
import os
import pandas as pd
import time
from tqdm import tqdm
from collections import defaultdict

# Add parent directory to path to import utils
sys.path.append('..')
from scripts.utils import (
    load_config, 
    get_openai_client, 
    call_llm, 
    load_prompt_template, 
    parse_json_response,
    save_csv_checkpoint,
    load_csv_checkpoint
)

# Enable autoreload for development
%load_ext autoreload
%autoreload 2

## Load Configuration and Data

In [ ]:
# Load configuration
config = load_config()
print(f"Loaded config for domain: {config['domain']['name']}")

# Load papers data
papers_df = load_csv_checkpoint("papers.csv")
print(f"Loaded {len(papers_df)} papers from papers.csv")

# Display sample of papers data
papers_df.head()

## Initialize LLM Client

In [ ]:
# Initialize OpenAI client for OpenRouter
client = get_openai_client(config)
print("LLM client initialized successfully")

# Test the client with a simple call
test_response = call_llm(
    client=client,
    prompt="Hello, can you confirm you're working?",
    model=config['llm']['model'],
    temperature=0.1,
    max_tokens=50
)
print(f"Test response: {test_response}")

## Process Papers and Extract Researchers

For each paper, we'll:
1. Format the extraction prompt with paper details
2. Call the LLM to extract researcher information
3. Parse the JSON response
4. Collect all researchers with their paper associations

In [ ]:
def extract_researchers_from_paper(paper_row, client, config):
    """Extract researcher information from a single paper using LLM"""
    
    # Load and format the prompt template
    prompt_template = load_prompt_template("extract_researcher_info")
    prompt = prompt_template.format(
        title=paper_row['title'],
        authors=paper_row['authors_raw'],
        abstract=paper_row['summary'],
        arxiv_id=paper_row['arxiv_id'],
        published_date=paper_row['published_date']
    )
    
    # Call LLM
    response = call_llm(
        client=client,
        prompt=prompt,
        model=config['llm']['model'],
        temperature=config['llm']['temperature'],
        max_tokens=config['llm']['max_tokens']
    )
    
    # Parse JSON response
    try:
        data = parse_json_response(response)
        researchers = data.get('researchers', [])
        
        # Add paper association to each researcher
        for researcher in researchers:
            researcher['papers'] = paper_row['arxiv_id']
            
        return researchers
        
    except Exception as e:
        print(f"Error parsing response for paper {paper_row['arxiv_id']}: {e}")
        print(f"Raw response: {response[:500]}...")
        return []

In [ ]:
# Process papers in batches to manage API calls
batch_size = config['processing']['batch_size']
max_researchers = config['processing']['max_researchers']

all_researchers = []
processed_count = 0

print(f"Processing up to {min(len(papers_df), max_researchers)} papers in batches of {batch_size}...")

# Limit processing for testing
papers_to_process = papers_df.head(max_researchers)

for i in tqdm(range(0, len(papers_to_process), batch_size), desc="Processing batches"):
    batch = papers_to_process.iloc[i:i+batch_size]
    
    for _, paper in batch.iterrows():
        researchers = extract_researchers_from_paper(paper, client, config)
        all_researchers.extend(researchers)
        processed_count += 1
        
        # Small delay between papers to be respectful
        time.sleep(0.5)
    
    # Progress update
    print(f"Processed {processed_count} papers, extracted {len(all_researchers)} researchers so far")

print(f"\nCompleted processing. Total researchers extracted: {len(all_researchers)}")

## Aggregate and Deduplicate Researchers

Since researchers may appear in multiple papers, we need to:
1. Group by researcher name
2. Aggregate their papers (semi-colon separated)
3. Keep the most complete information for each researcher

In [ ]:
# Convert to DataFrame for easier processing
researchers_df = pd.DataFrame(all_researchers)

if len(researchers_df) == 0:
    print("No researchers extracted. Please check the LLM responses above.")
else:
    print(f"Raw researchers data shape: {researchers_df.shape}")
    print(f"Columns: {list(researchers_df.columns)}")
    researchers_df.head()

In [ ]:
def aggregate_researchers(researchers_df):
    """Aggregate researchers by name, combining papers and keeping best info"""
    
    if researchers_df.empty:
        return pd.DataFrame(columns=['name', 'affiliation', 'research_focus', 'seniority', 'papers'])
    
    # Group by name (case-insensitive)
    aggregated = defaultdict(lambda: {
        'name': '',
        'affiliation': 'Unknown',
        'research_focus': '',
        'seniority': 'mid',  # default
        'papers': []
    })
    
    for _, researcher in researchers_df.iterrows():
        name_key = researcher['name'].strip().lower()
        
        # Update name (keep original casing)
        if not aggregated[name_key]['name']:
            aggregated[name_key]['name'] = researcher['name'].strip()
        
        # Update affiliation if we have one (prefer non-Unknown)
        if researcher.get('affiliation', 'Unknown') != 'Unknown' and aggregated[name_key]['affiliation'] == 'Unknown':
            aggregated[name_key]['affiliation'] = researcher['affiliation']
        
        # Update research focus if we have one
        if researcher.get('research_focus', '') and not aggregated[name_key]['research_focus']:
            aggregated[name_key]['research_focus'] = researcher['research_focus']
        
        # Update seniority
        if researcher.get('seniority'):
            aggregated[name_key]['seniority'] = researcher['seniority']
        
        # Add paper to list
        if researcher.get('papers'):
            aggregated[name_key]['papers'].append(researcher['papers'])
    
    # Convert back to DataFrame
    result_data = []
    for data in aggregated.values():
        # Remove duplicates from papers list and join with semicolons
        unique_papers = list(set(data['papers']))
        data['papers'] = ';'.join(sorted(unique_papers))
        result_data.append(data)
    
    return pd.DataFrame(result_data)

# Aggregate the researchers
final_researchers_df = aggregate_researchers(researchers_df)

print(f"After aggregation: {len(final_researchers_df)} unique researchers")
print(f"\nSample of aggregated researchers:")
final_researchers_df.head(10)

## Validate and Save Output

Check the final data quality and save to CSV.

In [ ]:
# Validate the output
print("=== Output Validation ===")
print(f"Total researchers: {len(final_researchers_df)}")
print(f"Columns: {list(final_researchers_df.columns)}")

# Check for missing values
missing_data = final_researchers_df.isnull().sum()
print(f"\nMissing values per column:")
print(missing_data)

# Check seniority distribution
if 'seniority' in final_researchers_df.columns:
    seniority_counts = final_researchers_df['seniority'].value_counts()
    print(f"\nSeniority distribution:")
    print(seniority_counts)

# Sample some entries to verify quality
print(f"\n=== Sample Researchers ===")
for i, row in final_researchers_df.head(5).iterrows():
    print(f"\n{i+1}. {row['name']}")
    print(f"   Affiliation: {row['affiliation']}")
    print(f"   Seniority: {row['seniority']}")
    print(f"   Papers: {row['papers']}")
    print(f"   Research focus: {row['research_focus'][:100]}..." if len(str(row['research_focus'])) > 100 else f"   Research focus: {row['research_focus']}")

In [ ]:
# Save the final researchers data
save_csv_checkpoint(final_researchers_df, "researchers.csv")
print(f"\n✅ Successfully saved researchers.csv with {len(final_researchers_df)} researchers")

## Summary

This notebook successfully:
- Processed papers from `data/papers.csv`
- Used LLM to extract researcher information from each paper
- Aggregated researchers across multiple papers
- Saved deduplicated results to `data/researchers.csv`

The output includes:
- Researcher names
- Affiliations (when available)
- Research focus summaries
- Seniority levels (junior/mid/senior)
- Associated paper IDs

Next step: Notebook 04 will enrich these researchers with social media profiles.